# 04 · Retorno R3: predicción prenatal, población y auditoría de conteos — Muerte fetal, MIMIC-IV (M3 · R3)

| Campo | Contenido |
|---|---|
| Secuencia del framework | 01: E1 · E2 · E3 · P1 → 02: E4 → 03: E5 · E6 → R3 → **04: R3 · E2 · E3 · P1 · E4** → 05: E5 · E6 · E7 · P2 → 06: P3 · clase · E11 |
| Qué hace | Aplica el retorno R3 (otro punto de predicción): con lo que se sabía de la mujer **antes** del ingreso para el parto, ¿se puede ver un riesgo alto de muerte fetal? Construye o verifica el universo obstétrico y la etiqueta de este punto de predicción (E2, E3), reproduce los conteos de referencia (P1) y audita cuánta información existe antes del ingreso (E4) |
| Etapas y puertas | R3 · E2 · E3 · P1 · E4 |
| Decisiones | Punto de predicción en el ingreso con la información registrada al menos 7 días antes; predictores fijados con las frecuencias de toda la población; ver `docs/outcome_card_antenatal_es.md` |
| Entradas | Tablas hosp de MIMIC-IV (`admissions`, `patients`, `diagnoses_icd`, `procedures_icd`, `omr`, `labevents`, `d_labitems`) |
| Tablas del esquema mimiciv | `antenatal_obstetric_universe` (`sql/03_antenatal/01_obstetric_universe_es.sql`) y `antenatal_fetal_death_label` (`sql/03_antenatal/02_fetal_death_label_es.sql`) |
| Script | `sql/03_antenatal/03_prior_information_es.sql` (solo lectura) |
| Salidas | `data/outputs_04_antenatal_count_audit.txt` (solo conteos agregados; de 1 a 4 se muestran como "<5") |
| Tiempo | Varios minutos: la auditoría lee `labevents` de las pacientes de la población |
| Anterior / siguiente | `03_unit_and_prediction_point` / `05_antenatal_predictors` |
| Versión en inglés | `04_antenatal_count_audit.ipynb` (mismo código) |

## R3 · Retorno: otro punto de predicción

El notebook 03 encontró que más de la mitad de las muertes fetales ya estaban presentes al ingreso + 6 h, así que el modelo en ese punto no se ajustó. El retorno R3 traslada la pregunta a un momento en el que la muerte todavía no había ocurrido. **Punto de predicción:** la hora del ingreso, usando solo la información registrada **al menos 7 días antes** (admisiones anteriores, mediciones ambulatorias y laboratorios ambulatorios); 1 y 30 días como sensibilidad. El margen de 7 días deja fuera lo registrado después de una muerte diagnosticada en consulta, que suele seguirse de un ingreso en pocos días. El desenlace, la población y la unidad no cambian; el universo y la etiqueta se reconstruyen desde las tablas de MIMIC-IV, para que este punto de predicción no dependa de ninguna tabla del anterior.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| El modelo del primer punto de predicción no puede ajustarse (C1 anticipado) y queda un retorno disponible | R3: adelantar el punto de predicción a antes del ingreso | 1 de los 2 retornos permitidos |

> **Reconstruir las tablas.** Las tablas del esquema `mimiciv` que usa este notebook se leen como están y solo se corren sus verificaciones. Si una tabla no existe, se construye con su script SQL. Para reconstruirla (por ejemplo, si cambia un script de `sql/`), poner **`REBUILD = True`** en la primera celda de código y volver a correr el notebook.

In [1]:
# Setup: paths, language, presentation helpers and the database connection
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "es"
import importlib
import libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner
for _mod in (libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner):
    importlib.reload(_mod)                       # a running kernel picks up changes of the libraries
from libraries.presentation import setup, tr, say, show, show_results, check, decision, notebook_summary, write_outputs
from libraries.sql_runner import run_script, ensure_table, table_exists
from libraries.database import get_connect_database
setup(LANG)

REBUILD = False      # True: rebuild the mimiciv tables of this notebook from their SQL scripts
SQL = ROOT / "sql"
OUT_FILE = ROOT / "data" / "outputs_04_antenatal_count_audit.txt"
engine = get_connect_database()

## E2 · Universo y población del punto de predicción prenatal

El universo obstétrico (admisiones de mujeres de 12 a 55 años con un diagnóstico de embarazo, parto o puerperio) y la marca de parto se reconstruyen aquí desde las tablas de MIMIC-IV. La población es la misma del notebook 01: hospitalizaciones de parto más las admisiones con un código de mortinato fuera de la marca de parto.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Los conteos reproducen el notebook 01 | Se pasa a E3 | 25.736 admisiones; población 19.440 |
| Difieren | Revisar `sql/03_antenatal/01_obstetric_universe_es.sql` | — |

In [2]:
# Stage S2: obstetric universe of the antenatal prediction point: mimiciv.antenatal_obstetric_universe
resu = ensure_table(engine, "antenatal_obstetric_universe", SQL / "03_antenatal" / "01_obstetric_universe.sql", REBUILD, log=say)
show_results(resu, ["Q-U1 reference counts", "Q-U2 population by evidence and era"])

01_obstetric_universe.sql: 3 sentencias corridas (solo verificaciones), 10 omitidas, 0 errores, 0.1 s


,admisiones obstétricas,pacientes,hospitalizaciones de parto,con código de mortinato,población
1,"25,736","15,084","19,425",124,"19,440"


,era CIE,partos,código de mortinato sin marca de parto
1,solo CIE-10,"8,883",10
2,solo CIE-9,"10,542",5


## E3 · Definición del desenlace — la misma etiqueta, reconstruida para este punto de predicción

Definición estricta (muerte fetal de todo el embarazo: CIE-10-CM O36.4, Z37.1, Z37.4, Z37.7; CIE-9-CM 656.4, V27.1, V27.4, V27.7) como etiqueta principal; las demás definiciones como sensibilidad, como en la decisión D5.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| El catálogo tiene los 18 patrones y los tamaños de la etiqueta coinciden con el notebook 01 | Se pasa a P1 | 129 eventos estrictos; 142 con códigos parciales |
| Difieren | Revisar `sql/03_antenatal/02_fetal_death_label_es.sql` | — |

In [3]:
# Stage S3: label by admission of the antenatal prediction point: mimiciv.antenatal_fetal_death_label
resl = ensure_table(engine, "antenatal_fetal_death_label", SQL / "03_antenatal" / "02_fetal_death_label.sql", REBUILD, log=say)
show_results(resl, ["Q-L1 code patterns by tier", "Q-L3 label sizes"])

02_fetal_death_label.sql: 4 sentencias corridas (solo verificaciones), 7 omitidas, 0 errores, 0.2 s


,nivel,versión CIE,patrones
1,excluido,9,<5
2,excluido,10,<5
3,parcial,9,<5
4,parcial,10,<5
5,estricto,9,<5
6,estricto,10,<5


,principal (estricta),estricta + parcial,estricta sin código de pérdida temprana,22 semanas o más,28 semanas o más,estrictos sin semanas,estrictos con código del registro del recién nacido
1,129,142,119,55,37,61,0


## P1 · Puerta de definición — la misma población y los mismos eventos del notebook 01

Los conteos de referencia son los del notebook 01 y de la auditoría de conteos. Si alguno difiere, el notebook se detiene.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Se reproducen todos los conteos | P1 pasa: se pasa a la auditoría de la información previa al ingreso (E4) | Pasa |
| Un conteo difiere | Se detiene (`assert`): volver a E2 o E3 | — |

In [4]:
# Gate G1: reference counts (notebook 01 and the count audit)
u1, l2 = resu["Q-U1 reference counts"].iloc[0], resl["Q-L2 reference counts"].iloc[0]
l3 = resl["Q-L3 label sizes"].iloc[0]
g1_ok = check([
    ("obstetric admissions", int(u1.n_obstetric_admissions), 25736),
    ("patients of the universe", int(u1.n_patients), 15084),
    ("delivery hospitalizations", int(u1.n_delivery_hospitalizations), 19425),
    ("population (delivery flag or stillbirth code)", int(u1.n_population), 19440),
    ("population: admissions", int(l2.n_admissions), 19440),
    ("population: patients", int(l2.n_patients), 13735),
    ("strict events (main label)", int(l2.n_strict), 129),
    ("events with partial codes", int(l2.n_events_any), 142),
    ("strict events without an early-loss code", int(l3.n_strict_without_early_loss_code), 119),
    ("strict events with 22 weeks or more", int(l3.n_ge22_weeks), 55),
    ("strict events with 28 weeks or more", int(l3.n_ge28_weeks), 37),
], "G1 · Reference counts")
decision("G1 · Definition gate", [
    ("population, unit and outcome of notebook 01, rebuilt", "passes" if g1_ok else "DOES NOT PASS",
     "on to S4" if g1_ok else "stop: go back to S2 or S3")])
assert g1_ok, "gate G1 does not pass: a reference count differs"

,conteo,cargado,esperado,resultado
1,admisiones obstétricas,"25,736","25,736",reproducido
2,pacientes del universo,"15,084","15,084",reproducido
3,hospitalizaciones de parto,"19,425","19,425",reproducido
4,población (marca de parto o código de mortinato),"19,440","19,440",reproducido
5,población: admisiones,"19,440","19,440",reproducido
6,población: pacientes,"13,735","13,735",reproducido
7,eventos estrictos (etiqueta principal),129,129,reproducido
8,eventos con códigos parciales,142,142,reproducido
9,eventos estrictos sin código de pérdida temprana,119,119,reproducido
10,eventos estrictos con 22 semanas o más,55,55,reproducido


,revisión,resultado,decisión
1,"población, unidad y desenlace del notebook 01, reconstruidos",pasa,se pasa a E4


## E4 · Auditoría de conteos — ¿cuánto se sabe antes del ingreso? (factibilidad)

Tres preguntas antes de cualquier modelo: cuántas mujeres tienen alguna información antes del ingreso (si casi ninguna, el modelo no es factible); qué información hay, con qué frecuencia y qué tan reciente; y, con eso, qué predictores entran. **Regla contra la selección por desenlace:** solo la disponibilidad de bloques completos (Q-A1 a Q-A3) se separa entre eventos y no eventos, porque decide la factibilidad; la frecuencia de cada condición o medición (Q-A4 a Q-A7) se muestra solo para toda la población.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| La mayoría de los eventos tiene alguna información antes del ingreso | El modelo es factible: se pasa a los predictores (notebook 05) | 99 de 129 eventos (76,7%) con el margen de 7 días |
| Casi ningún evento tiene información | No es factible: siguiente peldaño de la escalera (R1) u otro retorno | — |
| Condiciones raras (menos del 1% de la población) | Se unen en compuestos, nunca elegidas por su asociación con el desenlace | Dos compuestos: resultado adverso en embarazo anterior; otra condición crónica de alto riesgo |

In [5]:
# Stage S4: information available before the admission (read-only script; several minutes)
pri = run_script(engine, SQL / "03_antenatal" / "03_prior_information.sql", log=say)
show_results(pri, ["Q-A0 population check", "Q-A1 information available before admission", "Q-A2 information by period (7 days)",
                   "Q-A3 earlier admissions (7 days)", "Q-A4 conditions in earlier admissions (7 days)",
                   "Q-A5 outpatient measurements (7 days)", "Q-A6 outpatient laboratory values (7 days)",
                   "Q-A7 registration variables"])
qa1 = pri["Q-A1 information available before admission"]
ev7 = qa1[(qa1.buffer_days == 7) & (qa1.ev_group == "1 strict event")].iloc[0]
ne7 = qa1[(qa1.buffer_days == 7) & (qa1.ev_group == "3 non-event")].iloc[0]
feasible = ev7.n_any_information / ev7.n_admissions >= 0.5
decision("S4 · Feasibility and predictors", [
    ("strict events with any information (7-day margin)", f"{int(ev7.n_any_information)} of {int(ev7.n_admissions)} ({float(ev7.pct_any_information):.1f}%)",
     "the model is feasible" if feasible else "not feasible"),
    ("non-events with any information (7-day margin)", f"{float(ne7.pct_any_information):.1f}%",
     "each block carries an indicator: no record does not mean no condition"),
    ("margins of 1 and 30 days", "almost the same coverage (Q-A1)", "7 days main; 1 and 30 days as sensitivity"),
    ("earlier admissions (Q-A3)", "most were more than a year before (earlier pregnancies)",
     "two indicators: any earlier admission; admission in the 90 days before"),
    ("conditions in earlier admissions (Q-A4)", "frequent ones alone; rare ones in two composites",
     "9 parameters of history (outcome card, section 4)"),
    ("outpatient measurements (Q-A5)", "about 70% with weight and blood pressure, a median of about 2 weeks before",
     "last systolic and diastolic, any pressure of 140/90 or more, last body mass index (280 to 7 days before)"),
    ("outpatient laboratory values (Q-A6)", "hemoglobin in about 70%; creatinine old; A1c ordered selectively",
     "last hemoglobin only; one indicator of prenatal care in this hospital"),
    ("registration variables (Q-A7)", "almost complete", "age, race and ethnicity, insurance, language, marital status; admission type and location excluded (G2)"),
    ("candidate parameters", "27", "reduced set of 17 fixed before any model, for gate G3")])

03_prior_information.sql: 32 sentencias corridas (solo verificaciones), 0 omitidas, 0 errores, 147.4 s


,admisiones,pacientes,estrictos,eventos con parciales
1,"19,440","13,735",129,142


,margen (días),grupo,admisiones,alguna información,% con información,hospitalización anterior,hospitalización obstétrica anterior,mediciones ambulatorias,laboratorios ambulatorios
1,1,1 evento estricto,129,100,77.5,50,48,87,80
2,1,2 evento solo parcial,13,11,84.6,7,7,10,10
3,1,3 no evento,"19,298","17,174",89,"8,508","7,592","14,192","14,102"
4,7,1 evento estricto,129,99,76.7,49,47,86,79
5,7,2 evento solo parcial,13,11,84.6,7,7,10,10
6,7,3 no evento,"19,298","17,056",88.4,"8,369","7,438","14,151","13,965"
7,30,1 evento estricto,129,97,75.2,47,44,84,78
8,30,2 evento solo parcial,13,11,84.6,7,7,10,10
9,30,3 no evento,"19,298","16,811",87.1,"8,031","7,059","14,004","13,721"


,era CIE,grupo de año ancla,admisiones,alguna información,estrictos,estrictos con información
1,solo CIE-10,2008 - 2010,"1,533","1,497",12,12
2,solo CIE-10,2011 - 2013,"1,512","1,480",10,10
3,solo CIE-10,2014 - 2016,"2,561","2,342",16,15
4,solo CIE-10,2017 - 2019,"2,353","2,009",22,11
5,solo CIE-10,2020 - 2022,934,765,11,9
6,solo CIE-9,2008 - 2010,"6,700","5,910",34,30
7,solo CIE-9,2011 - 2013,"2,826","2,417",17,9
8,solo CIE-9,2014 - 2016,"1,020",746,7,<5
9,solo CIE-9,2017 - 2019,<5,0,0,0


,grupo,admisiones,ninguna,una,dos o tres,cuatro o más,última hace ≤ 90 días,91 a 365 días,más de 1 año
1,1 evento estricto,129,80,22,20,7,12,<5,34
2,2 evento solo parcial,13,6,<5,<5,0,<5,0,<5
3,3 no evento,"19,298","10,929","5,156","2,636",577,"1,513",870,"5,986"
4,,"19,440","11,015","5,181","2,660",584,"1,528",873,"6,024"


,antecedente,admisiones,% de la población,% de las que tienen historia
1,01 muerte fetal anterior,52,0.27,0.62
2,02 cesárea anterior,"1,670",8.59,19.82
3,03 parto pretérmino anterior,419,2.16,4.97
4,04 trastorno hipertensivo en embarazo anterior,989,5.09,11.74
5,05 diabetes gestacional anterior,418,2.15,4.96
6,06 hipertensión crónica,472,2.43,5.6
7,07 diabetes pregestacional,267,1.37,3.17
8,08 obesidad (código),669,3.44,7.94
9,09 enfermedad renal crónica,27,0.14,0.32
10,10 enfermedad tiroidea,728,3.74,8.64


,medición,admisiones con valor,% de la población,valor en el último año,mediana de días antes del ingreso
1,Weight (Lbs),"13,586",69.9,"11,124",13
2,Blood Pressure,"13,267",68.2,"12,161",12
3,BMI (kg/m2),"10,195",52.4,"7,950",14
4,Height (Inches),"9,227",47.5,"5,368",267
5,BMI,757,3.9,730,62
6,Weight,124,0.6,110,85
7,Blood Pressure Sitting,66,0.3,29,518
8,Blood Pressure Standing (1 min),52,0.3,22,518
9,Blood Pressure Lying,49,0.3,19,532
10,Blood Pressure Standing,10,0.1,<5,"1,139"


,examen,admisiones con valor,% de la población,valor en el último año,mediana de días antes del ingreso
1,cualquier otro examen,"14,054",72.3,"12,471",59
2,creatinina,"9,790",50.4,"6,460",200
3,hemoglobina,"13,453",69.2,"11,669",85
4,hemoglobina glicosilada (A1c),"3,349",17.2,"2,076",239


,admisiones,estado civil faltante,seguro faltante u otro,idioma faltante,raza o etnia desconocida
1,"19,440",385,441,30,928


,revisión,resultado,decisión
1,eventos estrictos con alguna información (margen de 7 días),99 de 129 (76.7%),el modelo es factible
2,no eventos con alguna información (margen de 7 días),88.4%,cada bloque lleva un indicador: no tener registro no significa no tener la condición
3,márgenes de 1 y 30 días,casi la misma cobertura (Q-A1),7 días principal; 1 y 30 días como sensibilidad
4,admisiones anteriores (Q-A3),la mayoría hace más de un año (embarazos anteriores),dos indicadores: alguna admisión anterior; admisión en los 90 días previos
5,antecedentes en admisiones anteriores (Q-A4),los frecuentes solos; los raros en dos compuestos,"9 parámetros de antecedentes (ficha del desenlace, sección 4)"
6,mediciones ambulatorias (Q-A5),"cerca del 70% con peso y presión arterial, una mediana de unas 2 semanas antes","última sistólica y diastólica, alguna presión de 140/90 o más, último índice de masa corporal (de 280 a 7 días antes)"
7,laboratorios ambulatorios (Q-A6),hemoglobina en cerca del 70%; creatinina antigua; A1c pedida de forma selectiva,solo la última hemoglobina; un indicador de control prenatal en este hospital
8,variables de registro (Q-A7),casi completas,"edad, raza y etnia, seguro, idioma, estado civil; tipo y lugar de admisión excluidos (P2)"
9,parámetros candidatos,27,"conjunto reducido de 17 fijado antes de cualquier modelo, para la puerta P3"


## Resumen del notebook

Indicadores principales de este notebook y el paso siguiente.

In [6]:
# Summary of notebook 04
notebook_summary([
    ("return used", "R3: antenatal prediction point (information at least 7 days before the admission)"),
    ("population / strict events", lambda: f"{int(l2.n_admissions):,} / {int(l2.n_strict)}"),
    ("gate G1", lambda: "passes" if g1_ok else "DOES NOT PASS"),
    ("strict events with information before the admission (7 days)", lambda: f"{int(ev7.n_any_information)} of {int(ev7.n_admissions)} ({float(ev7.pct_any_information):.1f}%)"),
    ("non-events with information before the admission (7 days)", lambda: f"{float(ne7.pct_any_information):.1f}%"),
    ("feasibility", lambda: "feasible" if feasible else "not feasible"),
    ("candidate parameters", "27 (reduced set: 17)"),
    ("next step", "predictor table, unit and leakage (S5, S6, S7, G2), notebook 05"),
])
write_outputs(OUT_FILE, "Notebook 04 (R3, S2, S3, G1, S4), fetal death MIMIC-IV, antenatal prediction point; aggregate output only")

,valor
indicador,
retorno usado,R3: punto de predicción prenatal (información de al menos 7 días antes del ingreso)
población / eventos estrictos,"19,440 / 129"
puerta P1,pasa
eventos estrictos con información antes del ingreso (7 días),99 de 129 (76.7%)
no eventos con información antes del ingreso (7 días),88.4%
factibilidad,factible
parámetros candidatos,27 (conjunto reducido: 17)
paso siguiente,"tabla de predictores, unidad y fuga (E5, E6, E7, P2), notebook 05"


salida agregada escrita: data/outputs_04_antenatal_count_audit.txt
